# Checkpoints


In [1]:
import random

class OrderGenerator:
    def __init__(self):
        self.order_id_int = 0
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = 1

    def generate(self):
        m = {
            "key": self.order_id_int,
            "value": {"id": self.order_id_int,
                      "product_id": random.randint(0, 100 - 1),
                      "customer_id": random.randint(0, 10 - 1),
                      "ts": self.ts_int},
        }
        #
        self.order_id_int += 1
        #
        self.ts_int += self.ts_step_int
        #
        return m

#

gen = OrderGenerator()
for _ in range(3):
    print(gen.generate())


{'key': 0, 'value': {'id': 0, 'product_id': 54, 'customer_id': 9, 'ts': 0}}
{'key': 1, 'value': {'id': 1, 'product_id': 32, 'customer_id': 2, 'ts': 1}}
{'key': 2, 'value': {'id': 2, 'product_id': 23, 'customer_id': 9, 'ts': 2}}


In [2]:
import sys
sys.path.insert(1, "..")

import kafi.streams.streams
import importlib
importlib.reload(kafi.streams.streams)

from kafi.kafka.cluster.cluster import Cluster
from kafi.streams.streams import Streams

import logging
logging.basicConfig(level=logging.DEBUG)

c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

source_str = "orders"
sink_str = "orders_aggregated"

tn = (
    Streams.source(c, source_str)
    
    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "order_ids": sorted(agg_r["order_ids"] + [r["id"]]),
                                    "product_ids": sorted(agg_r["product_ids"] + [r["product_id"]])},
                  {"orders": 0, "order_ids": [], "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "order_ids": agg_r["order_ids"],
                                     "product_ids": agg_r["product_ids"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r}).peek("sink")
    .sink(c, sink_str)
)

built_tn = Streams.build(tn)



In [3]:
from kafi.helpers import get_millis

orders_int = 1000

built_tn.reset()

checkpoint_str = "checkpoint"
g = f"group_{get_millis()}"

c.recreate(source_str)
c.recreate(sink_str)
c.recreate(checkpoint_str)

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, checkpoint_interval=0.01, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)
gen = OrderGenerator()

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



'orders'

DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1785157648746_checkpoint') offsets for topic 'checkpoint': {}


(['checkpoint'], 'group_1785157648746_checkpoint')


Consuming: 0 msg [00:00, ? msg/s]

DEBUG:kafi.streams.streams:Source consumer group ('group_1785157648746') offsets for topic 'orders': {}



(['orders'], 'group_1785157648746')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (29 KB compressed, 195 uncompressed).
INFO:kafi.streams.streams:Committed {0: 1000} for source orders.


sink: {'key': 4, 'value': {'customer_id': 4, 'orders': 96, 'order_ids': [0, 8, 24, 49, 64, 66, 68, 99, 103, 112, 125, 151, 153, 156, 160, 164, 170, 171, 183, 196, 231, 244, 248, 289, 294, 296, 305, 313, 324, 334, 348, 353, 356, 358, 362, 366, 368, 377, 447, 470, 478, 494, 501, 511, 529, 537, 543, 555, 594, 609, 615, 620, 621, 624, 652, 653, 675, 682, 683, 688, 691, 704, 710, 712, 714, 717, 722, 726, 740, 743, 744, 753, 784, 789, 791, 793, 818, 833, 836, 842, 848, 854, 875, 878, 899, 917, 920, 931, 940, 949, 952, 956, 965, 967, 980, 984], 'product_ids': [0, 1, 2, 2, 3, 3, 6, 6, 8, 10, 11, 12, 12, 12, 13, 13, 13, 15, 16, 16, 16, 19, 20, 21, 21, 22, 22, 22, 22, 23, 25, 26, 28, 30, 34, 34, 35, 35, 36, 37, 38, 39, 39, 39, 39, 40, 41, 41, 41, 41, 42, 46, 50, 50, 51, 52, 53, 55, 55, 55, 55, 56, 56, 58, 58, 59, 59, 62, 64, 66, 67, 69, 70, 70, 73, 73, 74, 75, 75, 76, 77, 77, 79, 80, 80, 80, 82, 82, 83, 90, 90, 92, 92, 94, 95, 96]}}
sink: {'key': 5, 'value': {'customer_id': 5, 'orders': 105, 'or

In [4]:
await stop_fun()
await Streams.tasks()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

In [ ]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



{'orders': 1000}
{'orders_aggregated': 10}
{'checkpoint': 31}


'orders'

DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1785157648746_checkpoint') offsets for topic 'checkpoint': {}


(['checkpoint'], 'group_1785157648746_checkpoint')


Consuming: 0 msg [00:00, ? msg/s]

INFO:kafi.streams.streams:Loading checkpoint...
INFO:kafi.streams.streams:...loading checkpoint done (29 KB compressed, 195 uncompressed).
DEBUG:kafi.streams.streams:Source consumer group ('group_1785157648746') offsets for topic 'orders': {0: 1000}
DEBUG:kafi.streams.streams:Source consumer group offsets for topic 'orders' overridden by checkpoint offsets: {0: 1000}



(['orders'], 'group_1785157648746')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (55 KB compressed, 413 uncompressed).
INFO:kafi.streams.streams:Committed {0: 2000} for source orders.


sink: {'key': 5, 'value': {'customer_id': 5, 'orders': 201, 'order_ids': [1, 6, 9, 13, 46, 52, 58, 75, 79, 82, 86, 100, 116, 126, 132, 135, 141, 144, 148, 186, 187, 201, 202, 208, 215, 227, 238, 239, 240, 252, 253, 269, 295, 298, 311, 312, 320, 322, 325, 329, 331, 349, 360, 371, 373, 388, 396, 401, 410, 421, 459, 460, 465, 473, 480, 485, 488, 490, 515, 533, 548, 549, 593, 604, 634, 636, 654, 669, 670, 679, 687, 692, 697, 729, 732, 757, 766, 778, 799, 809, 810, 813, 815, 822, 825, 849, 858, 872, 886, 888, 889, 897, 918, 932, 934, 943, 946, 955, 969, 970, 971, 972, 976, 977, 996, 1000, 1009, 1011, 1013, 1025, 1027, 1041, 1072, 1094, 1097, 1141, 1164, 1173, 1181, 1195, 1207, 1208, 1257, 1260, 1265, 1267, 1269, 1306, 1308, 1343, 1353, 1377, 1378, 1400, 1409, 1442, 1457, 1461, 1464, 1467, 1479, 1484, 1489, 1493, 1514, 1520, 1521, 1526, 1528, 1529, 1544, 1554, 1564, 1580, 1592, 1595, 1607, 1613, 1623, 1626, 1629, 1637, 1645, 1667, 1669, 1686, 1694, 1712, 1718, 1729, 1748, 1754, 1764, 1779, 1

In [6]:
await stop_fun()
await Streams.tasks()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

In [ ]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()


{'orders': 2000}
{'orders_aggregated': 20}
{'checkpoint': 88}


'orders'

DEBUG:kafi.streams.streams:Checkpoint consumer group ('group_1785157648746_checkpoint') offsets for topic 'checkpoint': {0: 31}


(['checkpoint'], 'group_1785157648746_checkpoint')


Consuming: 0 msg [00:00, ? msg/s]

INFO:kafi.streams.streams:Loading checkpoint...
INFO:kafi.streams.streams:...loading checkpoint done (55 KB compressed, 413 uncompressed).
DEBUG:kafi.streams.streams:Source consumer group ('group_1785157648746') offsets for topic 'orders': {0: 2000}
DEBUG:kafi.streams.streams:Source consumer group offsets for topic 'orders' overridden by checkpoint offsets: {0: 2000}



(['orders'], 'group_1785157648746')


INFO:kafi.streams.streams:Saving checkpoint...
INFO:kafi.streams.streams:...saving checkpoint done (71 KB compressed, 517 uncompressed).
INFO:kafi.streams.streams:Committed {0: 3000} for source orders.


sink: {'key': 2, 'value': {'customer_id': 2, 'orders': 287, 'order_ids': [30, 31, 32, 55, 76, 83, 88, 106, 108, 111, 115, 118, 123, 128, 143, 165, 174, 175, 184, 192, 198, 206, 209, 220, 241, 249, 260, 263, 267, 268, 270, 272, 276, 287, 292, 300, 303, 316, 328, 341, 370, 381, 399, 404, 406, 412, 427, 437, 449, 452, 462, 475, 482, 492, 499, 502, 553, 554, 560, 564, 566, 568, 571, 578, 580, 587, 591, 592, 595, 602, 612, 617, 619, 630, 641, 642, 646, 650, 689, 700, 718, 720, 724, 727, 730, 731, 747, 773, 780, 783, 786, 803, 816, 835, 855, 869, 870, 871, 909, 927, 962, 1001, 1006, 1016, 1053, 1064, 1067, 1069, 1099, 1101, 1105, 1107, 1109, 1110, 1112, 1118, 1125, 1131, 1163, 1168, 1177, 1182, 1201, 1209, 1210, 1218, 1242, 1244, 1256, 1292, 1302, 1311, 1323, 1328, 1340, 1345, 1354, 1357, 1358, 1372, 1387, 1391, 1418, 1432, 1438, 1448, 1458, 1497, 1501, 1502, 1503, 1510, 1513, 1522, 1542, 1561, 1569, 1570, 1581, 1582, 1596, 1604, 1617, 1622, 1653, 1658, 1660, 1672, 1683, 1687, 1693, 1696, 17

In [8]:
await stop_fun()
await Streams.tasks()

INFO:kafi.streams.streams:Safely stopping Streams...
INFO:kafi.streams.streams:...done.


[]

In [9]:
source_key_int_value_dict_dict = {}
source_m_list = c.cat(source_str)
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_order_id_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("order_ids", [])
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "order_ids": sorted(agg_order_id_int_list + [order_id_int]),
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
sink_m_list = c.cat(sink_str)
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["value"]["customer_id"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


(['orders'], '1785157718307')


Consuming: 0 msg [00:00, ? msg/s]


(['orders_aggregated'], '1785157723570')


Consuming: 0 msg [00:00, ? msg/s]


{4: {'customer_id': 4, 'orders': 313, 'order_ids': [0, 8, 24, 49, 64, 66, 68, 99, 103, 112, 125, 151, 153, 156, 160, 164, 170, 171, 183, 196, 231, 244, 248, 289, 294, 296, 305, 313, 324, 334, 348, 353, 356, 358, 362, 366, 368, 377, 447, 470, 478, 494, 501, 511, 529, 537, 543, 555, 594, 609, 615, 620, 621, 624, 652, 653, 675, 682, 683, 688, 691, 704, 710, 712, 714, 717, 722, 726, 740, 743, 744, 753, 784, 789, 791, 793, 818, 833, 836, 842, 848, 854, 875, 878, 899, 917, 920, 931, 940, 949, 952, 956, 965, 967, 980, 984, 1002, 1005, 1007, 1015, 1018, 1019, 1023, 1024, 1036, 1039, 1055, 1076, 1077, 1089, 1098, 1100, 1116, 1120, 1122, 1138, 1139, 1149, 1155, 1157, 1162, 1170, 1171, 1178, 1179, 1187, 1191, 1192, 1200, 1215, 1253, 1277, 1280, 1284, 1289, 1300, 1301, 1309, 1335, 1344, 1434, 1444, 1453, 1471, 1472, 1474, 1492, 1494, 1530, 1541, 1559, 1587, 1600, 1616, 1618, 1621, 1679, 1690, 1709, 1735, 1738, 1740, 1741, 1743, 1750, 1755, 1774, 1775, 1776, 1804, 1825, 1846, 1849, 1852, 1869, 187